# Graph link-prediction — `curated` dataset

> **Exploratory / rerun required.** Existing graph metrics use the old split and checkpoint-selection protocol and are not final benchmark results. See `README.md`.

Heterogeneous bipartite GNN/GAT (molecule ↔ protein), unentangled **XGBoost** probe on `[raw GIN mol ‖ graph-enriched ESM prot]`.

Two regimes: **transductive** (hold out edges ≈ stratified) and **inductive_molecule** (hold out whole molecules ≈ group_molecule). Grey bars = raw `ESM‖GIN→XGBoost` baseline (no graph).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

from orbind import hetero as H
from orbind.hetero_gat import HeteroGATLink
from orbind.dataset import metrics

In [ ]:
DATASET = "curated"   # "curated" | "full"
_CFG = {
    "curated": dict(
        pairs=ROOT/"data/processed/pairs_curated.csv",
        # full is the single source of truth; build_nodes() derives the 409
        # curated node set by filtering pairs_curated.csv receptors against it.
        prot=ROOT/"data/embeddings/proteins/esm2_650m_mean.npz",
        mol=ROOT/"data/embeddings/molecules/gin_supervised_contextpred_all_m2or.npz",
        ckpt_dir=ROOT/"results/curated/checkpoints",
        boost_csv=ROOT/"results/curated/tables/boost_results.csv",
    ),
    "full": dict(
        pairs=ROOT/"data/processed/pairs_m2or_full.csv",
        prot=ROOT/"data/embeddings/proteins/esm2_650m_mean.npz",
        mol=ROOT/"data/embeddings/molecules/gin_supervised_contextpred_all_m2or.npz",
        ckpt_dir=ROOT/"results/full/checkpoints",
        boost_csv=ROOT/"results/full/tables/boost_results.csv",
    ),
}
CFG = _CFG[DATASET]
print("DATASET =", DATASET, "| ckpt_dir =", CFG["ckpt_dir"])

In [ ]:
# Node features — deterministic, same ordering as during training
Xm, Xp, _, _, _ = H.build_nodes(CFG["pairs"], CFG["prot"], CFG["mol"])
x_dict = {H.MOL: Xm, H.PROT: Xp}

In [ ]:
# Load + evaluate all unentangled checkpoints (GNN and GAT)
def eval_unentangled(glob_pattern):
    records, preds = [], {}
    for path in sorted(CFG["ckpt_dir"].glob(glob_pattern)):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        if not ckpt.get("head", "").endswith("boost"):
            continue
        regime = ckpt["regime"]
        label  = ckpt["label"]
        _, test_y = ckpt["test_sup"]
        scores = ckpt["test_scores"]
        y = test_y.numpy()
        m = metrics(y, scores)
        records.append({"label": label, "regime": regime, **m})
        preds[(label, regime)] = (y, scores)
    df = (pd.DataFrame(records)
            .set_index(["regime", "label"])
            .sort_index()
            .round(3)) if records else pd.DataFrame()
    return df, preds

# XGBoost baseline (raw ESM, no GNN)
DOT = "·"
boost_raw = pd.read_csv(CFG["boost_csv"], index_col=0)
boost_col = {"transductive": "ESM" + DOT + "strat", "inductive_molecule": "ESM" + DOT + "mol"}

df_gnn, preds_gnn = eval_unentangled("gnn_*.pt")
df_gat, preds_gat = eval_unentangled("gat_*.pt")

print(f"GNN: {len(preds_gnn)} checkpoints,  GAT: {len(preds_gat)} checkpoints")
display(df_gnn)
display(df_gat)

In [ ]:
# ── shared plotting helper ─────────────────────────────────────────────────
PLOT_METRICS = ["AUROC", "AUPRC", "MCC", "F1"]
REGIMES      = ["transductive", "inductive_molecule"]

def bar_comparison(label_val_pairs, boost_raw, boost_col,
                   title, save_name, figsize_w=4.5):
    """
    label_val_pairs: list of (display_name, color, {regime: value})
    """
    all_names  = [n for n, _, _ in label_val_pairs] + ["XGBoost"]
    all_colors = [c for _, c, _ in label_val_pairs] + ["#A0A0A0"]

    fig, axes = plt.subplots(len(REGIMES), len(PLOT_METRICS),
                              figsize=(figsize_w * len(PLOT_METRICS), 3.5 * len(REGIMES)),
                              sharey=False)
    axes = np.array(axes).reshape(len(REGIMES), len(PLOT_METRICS))

    for ri, regime in enumerate(REGIMES):
        bcol = boost_col.get(regime)
        bval_row = {m: float(boost_raw.loc[m, bcol])
                    if bcol and bcol in boost_raw.columns else float("nan")
                    for m in PLOT_METRICS}
        for mi, metric in enumerate(PLOT_METRICS):
            ax = axes[ri, mi]
            vals = [val_dict.get((regime, metric), float("nan"))
                    for _, _, val_dict in label_val_pairs] + [bval_row[metric]]
            bars = ax.bar(all_names, vals, color=all_colors)
            ax.set_title(f"{regime}\n{metric}", fontsize=9)
            ax.set_ylim(0, 1)
            ax.tick_params(axis="x", labelsize=7, rotation=20)
            for bar, val in zip(bars, vals):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                            f"{val:.3f}", ha="center", va="bottom", fontsize=7)

    fig.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / f"notebooks/figures/{save_name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def df_vals(df, label):
    """Extract {(regime, metric): value} for one label from a df."""
    out = {}
    for regime in REGIMES:
        for metric in PLOT_METRICS:
            key = (regime, label)
            out[(regime, metric)] = df.loc[key, metric] if key in df.index else float("nan")
    return out

In [ ]:
# Plot B: MP mode comparison — Boost probe only (GNN)
bar_comparison(
    [
        ("pos_only",  "#4878CF", df_vals(df_gnn, "pos_only_unentangled_boost")),
        ("all_edges", "#6ACC65", df_vals(df_gnn, "all_edges_unentangled_boost")),
        ("signed",    "#D65F5F", df_vals(df_gnn, "signed_unentangled_boost")),
    ],
    boost_raw, boost_col,
    title="B: MP mode comparison — Boost probe, GNN  (grey = raw XGBoost baseline)",
    save_name="gnn_B_mp_modes",
)

In [ ]:
# Plot C: GAT vs GNN — signed, Boost probe
bar_comparison(
    [
        ("GNN signed · Boost", "#D65F5F", df_vals(df_gnn, "signed_unentangled_boost")),
        ("GAT signed · Boost", "#9467BD", df_vals(df_gat, "signed_unentangled_boost")),
    ],
    boost_raw, boost_col,
    title="C: GAT vs GNN — signed, Boost probe  (grey = raw XGBoost baseline)",
    save_name="gnn_C_gat_vs_gnn",
)

In [ ]:
# Plot D: Quality filter comparison — signed, Boost probe (GNN)
bar_comparison(
    [
        ("signed (all mols)", "#D65F5F", df_vals(df_gnn, "signed_unentangled_boost")),
        ("signed q=0.87",     "#B8479E", df_vals(df_gnn, "signed_q87_unentangled_boost")),
        ("signed q=0.95",     "#8B3A8B", df_vals(df_gnn, "signed_q95_unentangled_boost")),
        ("signed q=0.99",     "#4A1A4A", df_vals(df_gnn, "signed_q99_unentangled_boost")),
    ],
    boost_raw, boost_col,
    title="D: Quality filter — signed, Boost probe  (grey = raw XGBoost baseline)",
    save_name="gnn_D_q_filter",
)

In [ ]:
# Plot E: History (x0||x1||x2) vs final z_prot — GNN and GAT, signed, Boost probe
bar_comparison(
    [
        ("GNN · last",    "#D65F5F", df_vals(df_gnn, "signed_unentangled_boost")),
        ("GNN · history", "#FF9F7F", df_vals(df_gnn, "signed_unentangled_history_boost")),
        ("GAT · last",    "#9467BD", df_vals(df_gat, "signed_unentangled_boost")),
        ("GAT · history", "#C5A3E8", df_vals(df_gat, "signed_unentangled_history_boost")),
    ],
    boost_raw, boost_col,
    title="E: Concat all stages vs final stage — signed, Boost probe  (grey = raw XGBoost baseline)",
    save_name="gnn_E_history",
)